# Group Lab 3: Algorithm Development, Interpolation, and Smoothing

        **Week:** Week 9

        **Lab type:** Group lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Write pseudocode.
- Implement a moving average.
- Compare with pandas rolling calculations.
- Interpret smoothing effects.

        ## Earth and environmental motivation

        Writing an algorithm from scratch helps students see the assumptions hidden inside common data-analysis tools.

        ## Dataset

        Weather or streamflow time series

        ## Python concepts used

        - Pseudocode
- Moving windows
- Interpolation
- Smoothing
- Comparison tests

## Group Lab 2 Debrief and Collaborative Debugging (First 20 Minutes)

Open the debrief card from Group Lab 2. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. The class will investigate one open problem one check at a time.

- 0-3 min: review the issue board.
- 3-11 min: student reports.
- 11-18 min: collaborative debugging.
- 18-20 min: record one reusable lesson and connect it to today's Lab.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

## Group roles

- Module A: identify the computational problem and prepare data.
- Module B: implement the algorithm from scratch.
- Module C: compare outputs and interpret differences.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
flow = stream["discharge_cfs"].to_numpy()

In [ ]:
def moving_average(values, window):
    output = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        subset = values[start:i + 1]
        output.append(np.mean(subset))
    return np.array(output)

smooth_manual = moving_average(flow, 14)
smooth_pandas = stream["discharge_cfs"].rolling(14, min_periods=1).mean().to_numpy()
print("Maximum difference:", np.max(np.abs(smooth_manual - smooth_pandas)))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(stream["date"], flow, color="lightgray", linewidth=0.6, label="Daily")
ax.plot(stream["date"], smooth_manual, color="darkgreen", linewidth=1.0, label="14-day moving average")
ax.set_xlabel("Date")
ax.set_ylabel("Discharge (cfs)")
ax.set_title("Manual moving-average smoothing")
ax.legend()
fig.tight_layout()
plt.show()

## Write the plan before the code (Module A)

Before implementing, write pseudocode the whole group agrees on. Example for
a centered moving average:

```text
for each position i in the series:
    start = i minus half the window (not below 0)
    end   = i plus half the window (not past the end)
    output[i] = mean of values from start to end
```

Pseudocode exposes the two design decisions that matter here: how the window
is placed (centered or trailing) and what happens at the edges.

## Guided coding: centered vs trailing windows

The trailing window you built above only looks backward, so smoothed peaks
arrive late. A centered window looks both ways. Zoom into one flood to see
the timing difference; this matters whenever event timing is the science
question.

In [ ]:
def moving_average_centered(values, window):
    half = window // 2
    output = []
    for i in range(len(values)):
        start = max(0, i - half)
        end = min(len(values), i + half + 1)
        output.append(np.mean(values[start:end]))
    return np.array(output)

smooth_centered = moving_average_centered(flow, 14)

zoom = slice(1900, 2100)
dates_zoom = stream["date"].iloc[zoom]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates_zoom, flow[zoom], color="lightgray", linewidth=0.8, label="Daily")
ax.plot(dates_zoom, smooth_manual[zoom], color="darkgreen", label="Trailing 14-day")
ax.plot(dates_zoom, smooth_centered[zoom], color="purple", label="Centered 14-day")
ax.set_xlabel("Date")
ax.set_ylabel("Discharge (cfs)")
ax.set_title("Trailing windows lag the peaks; centered windows do not")
ax.legend()
fig.tight_layout()
plt.show()

## Guided coding: fill a sensor gap by hand (Module B)

Sensors fail. The function below fills a known gap by connecting the last
good value before the gap to the first good value after it. Because we
created the gap ourselves, we can score the fill against the truth.

In [ ]:
def fill_gap_linear(values, gap_start, gap_length):
    """Fill values[gap_start : gap_start + gap_length] by connecting the endpoints."""
    filled = values.copy()
    left = values[gap_start - 1]
    right = values[gap_start + gap_length]
    for step in range(gap_length):
        fraction = (step + 1) / (gap_length + 1)
        filled[gap_start + step] = left + fraction * (right - left)
    return filled

gap_start = 2000
gap_length = 7
true_segment = flow[gap_start : gap_start + gap_length].copy()
with_gap = flow.copy()
with_gap[gap_start : gap_start + gap_length] = np.nan

filled = fill_gap_linear(with_gap, gap_start, gap_length)
print("True values:  ", np.round(true_segment, 0))
print("Filled values:", np.round(filled[gap_start : gap_start + gap_length], 0))

In [ ]:
from earthcourse.stats import rmse

gap_results = []
for gap_length in [3, 7, 14, 30]:
    true_segment = flow[gap_start : gap_start + gap_length]
    with_gap = flow.copy()
    with_gap[gap_start : gap_start + gap_length] = np.nan
    filled = fill_gap_linear(with_gap, gap_start, gap_length)
    error = rmse(true_segment, filled[gap_start : gap_start + gap_length])
    gap_results.append({"gap_days": gap_length, "rmse_cfs": round(error, 1)})
print(pd.DataFrame(gap_results))

pandas_filled = pd.Series(with_gap).interpolate(method="linear").to_numpy()
print("Maximum difference from pandas interpolate on the 30-day gap:",
      f"{np.nanmax(np.abs(pandas_filled - filled)):.6f}")

The error grows with gap length because a straight line cannot recover a
storm that happened inside the gap. The pandas comparison should agree with
your implementation almost exactly; that agreement is your correctness test
(Module C).

## Try it yourself

Change the window length and explain what detail is lost or made clearer.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Create artificial gaps of at least three different lengths in the discharge record. Fill each gap with your interpolation algorithm, calculate RMSE against the hidden observations, and plot RMSE versus gap length. Explain the observed pattern.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Implement a moving MEDIAN filter and compare it with the moving mean on a
   stretch that contains a sharp flood peak. Which filter preserves the peak
   better, and which suppresses single-day spikes better?
2. Quantify the edge effect: for a trailing 14-day window, how many values at
   the start of the series were computed from fewer than 14 points?
3. For windows of 7, 30, and 90 days, state in one sentence each what
   hydrologic signal survives the smoothing (storms, seasons, wet years).

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Pseudocode
- [ ] Moving average function
- [ ] Comparison with package result
- [ ] Interpretation

        ## Short reflection

        Why should we compare a from-scratch algorithm with a trusted implementation?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
